In [ ]:
# ===========================================
# Notebook Name:
# 06_migrate_scrape_log_columns
#
# Purpose:
# Rename pokemon.bronze.scrape_log.run_id to
# request_id, and add a new pipeline_run_id
# column, so the column names match the v2 DDL
# in 00_migration/02_create_v2_tables and the
# updated ingest notebooks (01_ingest/*).
#
# This is a schema change on a live table shared
# by three ingest notebooks
# (00_ingest_tournament_list, 01_ingest_event_results,
# 02_ingest_decks) and is NOT part of the
# Daily/Weekly Workflow.
#
# Backfill:
# Existing rows have no pipeline_run_id (it did
# not exist before this migration), so the new
# column is backfilled from the renamed
# request_id for historical rows only. This is a
# reasonable approximation for old data (each
# historical row already represents one
# notebook-run's fetch attempt at the old,
# per-request run_id granularity) and does not
# need to be exact since pipeline_run_id is only
# used for cross-task tracing going forward.
#
# Run manually, once, before running any updated
# 01_ingest/* notebook against this table.
# ===========================================

In [ ]:
SCRAPE_LOG_TABLE = "pokemon.bronze.scrape_log"

existing_columns = {
    field.name
    for field in spark.table(SCRAPE_LOG_TABLE).schema.fields
}

print("Existing columns:", sorted(existing_columns))

In [ ]:
# -------------------------------------------
# Delta column mapping must be enabled before
# RENAME COLUMN is allowed.
# -------------------------------------------
spark.sql(f"""
ALTER TABLE {SCRAPE_LOG_TABLE}
SET TBLPROPERTIES (
    'delta.columnMapping.mode' = 'name',
    'delta.minReaderVersion' = '2',
    'delta.minWriterVersion' = '5'
)
""")

print("delta.columnMapping.mode enabled on", SCRAPE_LOG_TABLE)

In [ ]:
# -------------------------------------------
# Rename run_id -> request_id, then add the new
# pipeline_run_id column. Each is its own
# ALTER TABLE statement so a partial failure
# leaves a clear error pointing at the specific
# step.
# -------------------------------------------
if "run_id" in existing_columns and "request_id" not in existing_columns:
    spark.sql(f"ALTER TABLE {SCRAPE_LOG_TABLE} RENAME COLUMN run_id TO request_id")
    print("Renamed run_id -> request_id")
else:
    print("Skipping rename: run_id absent or request_id already present.")

existing_columns_after_rename = {
    field.name
    for field in spark.table(SCRAPE_LOG_TABLE).schema.fields
}

if "pipeline_run_id" not in existing_columns_after_rename:
    spark.sql(f"ALTER TABLE {SCRAPE_LOG_TABLE} ADD COLUMNS (pipeline_run_id STRING)")
    print("Added column: pipeline_run_id")
else:
    print("Skipping add: pipeline_run_id already present.")

In [ ]:
# -------------------------------------------
# Backfill pipeline_run_id for historical rows
# that predate this column, using request_id as
# an approximation (see notebook header).
# -------------------------------------------
from pyspark.sql import functions as F

unbackfilled_count = (
    spark.table(SCRAPE_LOG_TABLE)
    .filter(F.col("pipeline_run_id").isNull())
    .count()
)

if unbackfilled_count > 0:
    spark.sql(f"""
    UPDATE {SCRAPE_LOG_TABLE}
    SET pipeline_run_id = request_id
    WHERE pipeline_run_id IS NULL
    """)
    print(f"Backfilled pipeline_run_id for {unbackfilled_count} historical row(s).")
else:
    print("No rows needed pipeline_run_id backfill.")

In [ ]:
# -------------------------------------------
# Verify: request_id and pipeline_run_id both
# exist and are never null; run_id no longer
# exists.
# -------------------------------------------
final_columns = {
    field.name
    for field in spark.table(SCRAPE_LOG_TABLE).schema.fields
}

if "run_id" in final_columns:
    raise ValueError("run_id still present after rename.")

for required_column in ("request_id", "pipeline_run_id"):
    if required_column not in final_columns:
        raise ValueError(f"{required_column} missing after migration.")

    null_count = (
        spark.table(SCRAPE_LOG_TABLE)
        .filter(F.col(required_column).isNull())
        .count()
    )

    if null_count > 0:
        raise ValueError(f"{required_column} has {null_count} null value(s) after migration.")

print("Verification passed: run_id renamed to request_id, pipeline_run_id backfilled.")
display(spark.sql(f"DESCRIBE TABLE {SCRAPE_LOG_TABLE}"))